In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import json
import pathlib
from functools import partial
from pathlib import Path

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 9})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.data_management import list_modeling_results, load_modeling_result, load_testing_sequences
from admpe.excitation.excitation_experiment_setup import setup_environment
from admpe.sets import SafelyExcitableSet
from admpe.vector_fields import CurvatureVectorField
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.evaluation.modeling_evaluation import (
    generate_default_prediction_validation_points,
    model_prediction,
    evaluate_model_rollout,
)

In [ ]:
OBJECTIVE_LABELS = {
    "dmpe_uniform": "Uniform",
    "dmpe_static_dynamics": "Dynamics",
    "dmpe_static_mixture_dynamics": "Mixture dynamics",
    "dmpe_static_curv": "Curvature",
    "dmpe_static_mixture_curv": "Mixture curvature",
}


def compute_fluid_level_diagnostics(points_per_dim=200, block_size=1024):
    # Same 200 x 200 grid and safely excitable support as the one-step evaluation.
    # Keep each sampled height separately instead of merging into 0.1 m bins.
    grid = np.linspace(-1.0, 1.0, points_per_dim)
    heights = 1.5 * (grid + 1.0)
    points = np.stack(np.meshgrid(grid, grid, indexing="ij"), axis=-1).reshape(-1, 2)
    height_ids = np.repeat(np.arange(points_per_dim), points_per_dim)

    safe_set, _ = SafelyExcitableSet.from_system_name("fluid_tank")
    keep = np.asarray(safe_set.contains(jnp.asarray(points, dtype=jnp.float32)))
    points, height_ids = points[keep], height_ids[keep]
    if not len(points):
        raise ValueError("No evaluation points belong to the safely excitable set.")
    counts = np.bincount(height_ids, minlength=points_per_dim)

    # Use float64 for the Hessian and the same grid resolution as the errors.
    with jax.enable_x64():
        env, penalty, _, _, _ = setup_environment("fluid_tank", "gru")
        vf = CurvatureVectorField.from_env(
            env, penalty, points_per_dim=points_per_dim, low=-1.0, high=1.0,
            use_discrete_dynamics=True, smooth_domain=lambda q: q[0] > -1.0,
        )
        z = np.asarray(vf.z)
        valid_curvature = np.asarray(safe_set.contains(vf.z)) & (z[:, 0] > -1.0)
        # Excluded points are missing, rather than artificial zero curvature.
        values = np.where(valid_curvature, np.asarray(vf.values)[:, 0], np.nan)
        curvature = pd.DataFrame({
            "height": np.repeat(heights, points_per_dim),
            "curvature": values,
        }).groupby("height")["curvature"].mean()
        # Retain the raw Hessian norm; the plot gives it its own right axis.

    rows = []
    selections = [
        ("GP", "gp_transition", "submission_nov", jnp.float64),
        ("MLP", "neural_euler_large_1", "submission_nov_1step", jnp.float32),
    ]
    for label, model_type, experiment, dtype in selections:
        entries = list_modeling_results(
            system_name="fluid_tank", model_type=model_type,
            exp_name=experiment, excitation_objective=list(OBJECTIVE_LABELS),
            status="completed",
        )
        if not entries:
            raise ValueError(f"No completed {label} runs found.")
        # Match the evaluation precision used for each model family.
        with jax.enable_x64(dtype == jnp.float64):
            for entry in tqdm(entries, desc=label):
                result = load_modeling_result(entry["path"], dtype=dtype)
                wrapped_model, wrapped_env = get_wrapped_model_and_env(result)
                errors = []
                for start in range(0, len(points), block_size):
                    block = jnp.asarray(points[start:start + block_size], dtype=dtype)
                    prediction, reference = model_prediction(
                        wrapped_model, wrapped_env, validation_points=block,
                    )
                    error = np.asarray(prediction, dtype=float) - np.asarray(reference, dtype=float)
                    errors.append((error ** 2).mean(axis=-1))
                squared_error = np.concatenate(errors)
                if not np.isfinite(squared_error).all():
                    raise ValueError(f"Nonfinite predictions: {entry['name']}")
                sse = np.bincount(
                    height_ids, weights=squared_error, minlength=points_per_dim,
                )
                for b, height in enumerate(heights):
                    rows.append({
                        "model": label, "exp_id": result["metadata"]["exp_id"],
                        "objective": result["metadata"]["excitation_objective"],
                        "height": height, "n": int(counts[b]), "sse": float(sse[b]),
                    })
                del result, wrapped_model, wrapped_env

    errors = pd.DataFrame(rows)
    for label, *_ in selections:
        found = set(errors.loc[errors["model"].eq(label), "objective"])
        missing = set(OBJECTIVE_LABELS).difference(found)
        if missing:
            raise ValueError(f"Missing {label} objectives: {sorted(missing)}")
    return errors, curvature

In [ ]:
fluid_level_errors, fluid_curvature = compute_fluid_level_diagnostics()

In [ ]:
from admpe.data_management import list_excitation_results

run = list_excitation_results(
    system_name="fluid_tank",
    model_type="perfect",
    exp_name="submission_nov",
    excitation_objective="dmpe_static_curv",
    status="completed",
)[0]

target_params = json.loads(
    (run["path"] / "experiment.json").read_text()
)["params"]["alg_params"]

n = target_params["points_per_dim"]
extent = target_params["grid_extend"]

fluid_curvature = pd.Series(
    np.asarray(target_params["target_distribution"]).reshape(n, -1).sum(axis=1),
    index=1.5 * (np.linspace(-extent, extent, n) + 1.0),
)
fluid_curvature = fluid_curvature.loc[
    (fluid_curvature.index >= 0.0)
]

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, ScalarFormatter

# Cell 1 already sets the documentation's 9 pt LaTeX fonts and 433.62 pt width.
data = fluid_level_errors.loc[
    fluid_level_errors["model"].isin(["GP", "MLP"])
    & fluid_level_errors["n"].gt(0)
].copy()
# Convert normalized-height squared error into physical units [m^2].
data["mse"] = 1.5**2 * data["sse"] / data["n"]
summary = (
    data.groupby(["model", "objective", "height"])["mse"]
    .agg(mean="mean", std="std")
    .reset_index()
)

colors = {
    objective: plt.get_cmap("tab10")(i) for i, objective in enumerate(OBJECTIVE_LABELS)
}

fig, axes = plt.subplots(
    1, 2, figsize=(figurewidth_in, 1.8), sharex=True, layout="constrained",
)
curvature_axes = []

finite_curvature = fluid_curvature.dropna()
curvature_max = float(finite_curvature.max())

if not np.isfinite(curvature_max) or curvature_max < 0:
    raise ValueError("No finite, nonnegative curvature values available.")

for ax, model in zip(axes, ["GP", "MLP"]):
    for objective, label in OBJECTIVE_LABELS.items():
        group = summary.loc[
            summary["model"].eq(model) & summary["objective"].eq(objective)
        ].sort_values("height")
        if group.empty:
            raise ValueError(f"No {model} results for {objective}.")
        x = group["height"].to_numpy()
        mean = group["mean"].to_numpy()
        std = group["std"].fillna(0).to_numpy()
        ax.plot(x, mean, color=colors[objective], linewidth=1.2)
        ax.fill_between(
            x, np.maximum(0, mean - std), mean + std,
            color=colors[objective], alpha=0.15, linewidth=0,
        )

    ax.set_title(model, fontsize=9, pad=3)
    ax.set_xlabel(r"Fluid level in $\mathrm{m}$", fontsize=9)
    ax.set_ylabel(r"One-step MSE in $\mathrm{m}^2$", fontsize=9)
    ax.set_xlim(0, 0.2)
    ax.set_xticks(np.arange(0, 0.21, 0.2))
    # Scale MSE to the displayed interval, independently of curvature.
    visible = summary.loc[summary["model"].eq(model) & summary["height"].between(0, 0.6)]
    upper = float((visible["mean"] + visible["std"].fillna(0)).max())
    ax.set_ylim(0, 1.05 * upper if upper > 0 else 1.0)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    formatter = ScalarFormatter(useMathText=True, useOffset=False)
    formatter.set_powerlimits((0, 0))
    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.get_offset_text().set_fontsize(8)
    ax.tick_params(axis="both", labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)

    curvature_ax = ax.twinx()
    curvature_axes.append(curvature_ax)
    curvature_ax.plot(
        fluid_curvature.index, fluid_curvature,
        color="black", linestyle="--", linewidth=1.0,
    )
    curvature_ax.set_ylim(0, 1.05 * curvature_max if curvature_max > 0 else 1.0)
    curvature_ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    curvature_ax.tick_params(axis="y", labelsize=8)

curvature_axes[-1].set_ylabel(r"$\bm{p}^*_\mathrm{curv}$", fontsize=9)
handles = [Line2D([], [], color=colors[o], linewidth=1.2, label=label)
           for o, label in OBJECTIVE_LABELS.items()]
handles.append(Line2D([], [], color="black", linestyle="--", linewidth=1.0,
                      label=r"$\bm{p}^*_\mathrm{curv}$ (right axis)"))
fig.legend(
    handles=handles, loc="outside lower center", ncols=6,
    frameon=True, fontsize=8, columnspacing=1.5, handletextpad=0.5,
)

fig.savefig("paper_fig/fluid_tank_errors.pdf", bbox_inches="tight")
fig.savefig("paper_fig/fluid_tank_errors.png", dpi=300, bbox_inches="tight")
plt.show()